<a href="https://colab.research.google.com/github/crislyn-bot/E2E-Transformer-Tboli-English-NMT/blob/main/mt_mbart_tboli_lang.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Note** This .ipnb file contains the training process of fine-tuning the *mt-mbart-50-rlhf-dpo-tboli-lang* as the machine translation (MT) component of Speech-to-text cascaded system for Tboli language

**THIS SOURCE CODE COVERS THE FOLLOWING PHASE:**


- MT fine-tuning using *mBart-50* as reference model (https://huggingface.co/facebook/mbart-large-50-many-to-many-mmt)
- Model Refining with RLHF-DPO (https://huggingface.co/papers/2305.18290)
- Final Model Evaluation

Final Model (Huggingface URL): https://huggingface.co/crslnjoyz/mt-mbart-50-rlhf-dpo-tboli-lang




# MT Model Fine-tuning Phase

In [ ]:
!pip install sacrebleu evaluate datasets transformers accelerate

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


!rm -rf /content/tboli_dataset  # Remove any previous incomplete or incorrect copy
!mkdir -p /content/tboli_dataset # Ensure the target directory exists
!cp -r /content/drive/MyDrive/tboli_dataset/* /content/tboli_dataset/ # Copy contents


In [ ]:
from huggingface_hub import logout; logout()

In [ ]:
# Hugging Face Login (for pushing model if desired)
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from datasets import Dataset, DatasetDict
import pandas as pd

df = pd.read_csv("/content/tboli_dataset/metadata_clean.csv")

mt_df = df[["tboli_script", "english_script"]].dropna()

train_df = df[df["split"] == "train"][["tboli_script", "english_script"]].dropna()
val_df = df[df["split"] == "val"][["tboli_script", "english_script"]].dropna()
test_df = df[df["split"] == "test"][["tboli_script", "english_script"]].dropna()

mt_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})



In [ ]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

model_name = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(
    model_name,
    src_lang="tl_XX",   # Tagalog as proxy for Tboli
    tgt_lang="en_XX"
)

model = MBartForConditionalGeneration.from_pretrained(model_name)


In [ ]:
def preprocess_mt(batch):
    model_inputs = tokenizer(
        batch["tboli_script"],
        truncation=True,
    )

    labels = tokenizer(
        batch["english_script"],
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


mt_dataset = mt_dataset.map(
    preprocess_mt,
    batched=True,
    remove_columns=mt_dataset["train"].column_names
)

In [ ]:
!pip install evaluate

In [ ]:
import evaluate

bleu = evaluate.load("bleu")
chrf = evaluate.load("chrf")

def compute_mt_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    refs = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    return {
        "bleu": bleu.compute(
            predictions=preds,
            references=[[r] for r in refs]
        )["bleu"] * 100,
        "chrf": chrf.compute(
            predictions=preds,
            references=refs
        )["score"]
    }

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt-pre-RLHF-v2",
    eval_strategy="steps",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    num_train_epochs=10,
    fp16=True,
    save_steps=500,
    eval_steps=500,
    logging_steps=100,
    predict_with_generate=True,
    generation_max_length=64,
    load_best_model_at_end=True,
    metric_for_best_model="chrf",
    greater_is_better=True,
    report_to="none",
    push_to_hub=True,  # Push to HF
    optim="adamw_torch" # Explicitly set optimizer type
)



In [ ]:
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq

# Define data collator for MT
data_collator_mt = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=mt_dataset["train"],
    eval_dataset=mt_dataset["validation"],
    data_collator=data_collator_mt, # Explicitly pass the data collator
    compute_metrics=compute_mt_metrics
)

# Training the MT model
trainer.train()

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("./mt-pre-RLHF-v2")
tokenizer.save_pretrained("./mt-pre-RLHF-v2")
print(" pre RLHF model saved to ./mt-pre-RLHF-v2")
trainer.push_to_hub()

# RLHF DPO Style Rating Sheets


> creating xlxs file as rating sheets for DPO training



In [ ]:
!pip install openpyxl sacrebleu evaluate -q

In [ ]:
import torch
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import (Font, PatternFill, Alignment, Border, Side)
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from datasets import Dataset, DatasetDict
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import evaluate, os

print("Imports done")

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


!rm -rf /content/tboli_dataset  # Remove any previous incomplete or incorrect copy
!mkdir -p /content/tboli_dataset # Ensure the target directory exists
!cp -r /content/drive/MyDrive/tboli_dataset/* /content/tboli_dataset/ # Copy contents


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("crslnjoyz/mt-pre-RLHF-v2")
model = AutoModelForSeq2SeqLM.from_pretrained("crslnjoyz/mt-pre-RLHF-v2")

In [ ]:
MODEL_PATH      = "crslnjoyz/mt-pre-RLHF-v2"           # saved MT model
DATASET_CSV     = "/content/tboli_dataset/metadata_clean.csv"
NUM_RATERS      = 10                         # set to your actual number of raters (10–15)
                                             # rows will be split evenly across raters
BATCH_SIZE      = 16
MAX_LEN         = 128
OUTPUT_XLSX     = "tboli_mt_for_rating.xlsx"
OUTPUT_CSV      = "tboli_mt_for_rating.csv"
RANDOM_SEED     = 42


In [ ]:
print("Loading model …")
tokenizer = MBart50TokenizerFast.from_pretrained(
    MODEL_PATH, src_lang="tl_XX", tgt_lang="en_XX"
)
model = MBartForConditionalGeneration.from_pretrained(MODEL_PATH)
model.generation_config.forced_bos_token_id = tokenizer.lang_code_to_id["en_XX"]
device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device).eval()
print(f" Model loaded on {device}")


In [ ]:
df_full  = pd.read_csv(DATASET_CSV)
val_df   = df_full[df_full["split"] == "val"][["tboli_script", "english_script"]].dropna()
val_df   = val_df.rename(columns={"tboli_script": "tboli_source",
                                   "english_script": "reference_english"})
sample_df = val_df.reset_index(drop=True)   # use ALL rows — no sampling

n_total       = len(sample_df)
rows_per_rater = int(np.ceil(n_total / NUM_RATERS))

# Assign a rater number (1-based) to each row for splitting into sheets later
sample_df["rater_assignment"] = [
    min((i // rows_per_rater) + 1, NUM_RATERS)
    for i in range(n_total)
]

print(f" Validation set: {n_total} rows  →  using ALL rows (no sampling)")
print(f"   Split across {NUM_RATERS} raters: ~{rows_per_rater} rows each")




In [ ]:
#  Run MT inference
def translate_batch(texts, batch_size=BATCH_SIZE):
    translations = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc   = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
        ).to(device)
        with torch.no_grad():
            out = model.generate(
                **enc,
                num_beams=4,
                max_length=MAX_LEN,
                early_stopping=True,
            )
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        translations.extend(decoded)
        print(f"  Translated {min(i+batch_size, len(texts))}/{len(texts)} …")
    return translations

print("Running inference …")
sample_df["mt_translation"] = translate_batch(sample_df["tboli_source"].tolist())
print(" Inference complete")

# Quick BLEU snapshot (vs reference)
bleu_metric = evaluate.load("bleu")
chrf_metric = evaluate.load("chrf")
bleu = bleu_metric.compute(
    predictions=sample_df["mt_translation"].tolist(),
    references=[[r] for r in sample_df["reference_english"].tolist()],
)["bleu"] * 100
chrf = chrf_metric.compute(
    predictions=sample_df["mt_translation"].tolist(),
    references=sample_df["reference_english"].tolist(),
)["score"]
print(f"\n  Pre-RLHF BLEU : {bleu:.2f}  |  chrF: {chrf:.2f}  (over all {n_total} val rows)")
print("  (Save these numbers — compare after DPO to measure improvement)\n")


In [ ]:
DARK  = "1A2E4A"
TEAL  = "2E86AB"
GOLD  = "F4A261"
LIGHT = "E8F4F8"
WHITE = "FFFFFF"
YELL  = "FFF9C4"
GREEN = "E8F5E9"
ORANG = "FFF3E0"

thin  = Side(style="thin",   color="CCCCCC")
med   = Side(style="medium", color=TEAL)
full_border = Border(top=thin, bottom=thin, left=thin, right=thin)

def styled(ws, row, col, value="", bg=WHITE, fg=DARK, bold=False,
           size=10, align="left", wrap=True, num_fmt=None, italic=False):
    c = ws.cell(row=row, column=col, value=value)
    c.font      = Font(name="Arial", bold=bold, color=fg, size=size, italic=italic)
    c.fill      = PatternFill("solid", fgColor=bg)
    c.alignment = Alignment(horizontal=align, vertical="center", wrap_text=wrap)
    c.border    = full_border
    if num_fmt:
        c.number_format = num_fmt
    return c

wb = Workbook()

# Sheet 1 MT Samples for Rating
ws = wb.active
ws.title = "MT Samples for Rating"
ws.sheet_view.showGridLines = False
ws.freeze_panes = "A5"

# Banner rows
ws.row_dimensions[1].height = 8
ws.row_dimensions[2].height = 40
ws.row_dimensions[3].height = 24
ws.row_dimensions[4].height = 38

def banner(ws, row, text, bg, fg=WHITE, size=13, bold=True, merge_to=9):
    c = ws.cell(row=row, column=1, value=text)
    c.font      = Font(name="Arial", bold=bold, color=fg, size=size)
    c.fill      = PatternFill("solid", fgColor=bg)
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.merge_cells(start_row=row, start_column=1,
                   end_row=row, end_column=merge_to)

banner(ws, 2, "TBOLI–ENGLISH MT  ·  RLHF RATING SHEET", DARK, size=15)
banner(ws, 3,
       f"Model: {MODEL_PATH}  |  ALL {n_total} val rows  "
       f"|  Pre-RLHF BLEU: {bleu:.2f}  chrF: {chrf:.2f}  "
       f"|  {NUM_RATERS} raters · ~{rows_per_rater} rows each",
       TEAL, size=10, bold=False)

# Column definitions: (header, width, bg)
COLS = [
    ("#",                          5,  DARK),
    ("TBOLI SOURCE\n(Input to MT)",  36, DARK),
    ("MT TRANSLATION\n(Model Output)", 36, DARK),
    ("REFERENCE\n(Ground Truth)",    28, DARK),
    ("ACCURACY\n(1–5)\n★ required",  13, DARK),
    ("FLUENCY\n(1–5)\n★ required",   13, DARK),
    ("CULTURAL FIT\n(1–5)\n★ required", 14, DARK),
    ("PREFERRED TRANSLATION\n(optional — your correction)", 36, DARK),
    ("COMMENTS\n(optional — notes on errors)", 36, DARK),
]

for j, (label, width, bg) in enumerate(COLS, 1):
    c = ws.cell(row=4, column=j, value=label)
    c.font      = Font(name="Arial", bold=True, color=WHITE, size=9)
    c.fill      = PatternFill("solid", fgColor=DARK)
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    c.border    = full_border
    ws.column_dimensions[get_column_letter(j)].width = width

# Data rows
# Score validation dropdown
dv = DataValidation(
    type="whole", operator="between",
    formula1=1, formula2=5,
    error="Enter 1–5 only",  errorTitle="Invalid Score",
    prompt="1=Very Poor  2=Poor  3=Acceptable  4=Good  5=Excellent",
    promptTitle="Rating Scale",
)
dv.sqref = f"E5:G{4 + n_total + 1}"
ws.add_data_validation(dv)

for i, row_data in sample_df.iterrows():
    r   = i + 5
    bg  = LIGHT if i % 2 == 0 else WHITE
    ws.row_dimensions[r].height = 60

    # #
    styled(ws, r, 1, i + 1, bg=bg, bold=True, align="center")
    # Tboli source
    styled(ws, r, 2, row_data["tboli_source"], bg=bg, italic=True)
    # MT translation
    styled(ws, r, 3, row_data["mt_translation"], bg=bg)
    # Reference / ground truth
    styled(ws, r, 4, row_data["reference_english"], bg=bg, fg="666666", italic=True)
    # Score cells (yellow = must fill)
    for col in [5, 6, 7]:
        styled(ws, r, col, "", bg=YELL, align="center", bold=True, size=12)
    # Preferred translation (light green)
    styled(ws, r, 8, "", bg=GREEN)
    # Comments (light orange)
    styled(ws, r, 9, "", bg=ORANG)

# Summary row
SR = n_total + 5
ws.row_dimensions[SR].height = 26
ws.merge_cells(start_row=SR, start_column=1, end_row=SR, end_column=4)
c = ws.cell(row=SR, column=1, value="COLUMN AVERAGES  (auto-calculated)")
c.font      = Font(name="Arial", bold=True, color=WHITE, size=10)
c.fill      = PatternFill("solid", fgColor=DARK)
c.alignment = Alignment(horizontal="center", vertical="center")

for col, col_letter in [(5, "E"), (6, "F"), (7, "G")]:
    formula = f"=IFERROR(AVERAGE({col_letter}5:{col_letter}{SR-1}),\"\")"
    c = ws.cell(row=SR, column=col, value=formula)
    c.font         = Font(name="Arial", bold=True, color=WHITE, size=11)
    c.fill         = PatternFill("solid", fgColor=TEAL)
    c.alignment    = Alignment(horizontal="center", vertical="center")
    c.number_format = "0.00"
    c.border        = full_border

for col in [8, 9]:
    styled(ws, SR, col, "", bg=DARK)

# Per-rater sheets: each rater gets their own tab
print(f"Building {NUM_RATERS} individual rater sheets …")
for rater_num in range(1, NUM_RATERS + 1):
    rater_rows = sample_df[sample_df["rater_assignment"] == rater_num]
    if rater_rows.empty:
        continue

    ws_r = wb.create_sheet(f"Rater {rater_num:02d}")
    ws_r.sheet_view.showGridLines = False
    ws_r.freeze_panes = "A5"

    ws_r.row_dimensions[1].height = 8
    ws_r.row_dimensions[2].height = 36
    ws_r.row_dimensions[3].height = 22
    ws_r.row_dimensions[4].height = 38

    banner(ws_r, 2,
           f"TBOLI–ENGLISH MT  ·  RATER {rater_num:02d}  "
           f"({len(rater_rows)} rows out of {n_total} total)",
           DARK, size=13)
    banner(ws_r, 3,
           "Fill in ACCURACY, FLUENCY, CULTURAL FIT (1–5) for every row.  "
           "Yellow cells = required.  Green = optional correction.  Orange = optional notes.",
           TEAL, size=9, bold=False)

    for j, (label, width, _) in enumerate(COLS, 1):
        c = ws_r.cell(row=4, column=j, value=label)
        c.font      = Font(name="Arial", bold=True, color=WHITE, size=9)
        c.fill      = PatternFill("solid", fgColor=DARK)
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        c.border    = full_border
        ws_r.column_dimensions[get_column_letter(j)].width = width

    # Score dropdown for this rater's sheet
    n_r = len(rater_rows)
    dv_r = DataValidation(
        type="whole", operator="between", formula1=1, formula2=5,
        error="Enter 1–5 only", errorTitle="Invalid Score",
        prompt="1=Very Poor  2=Poor  3=Acceptable  4=Good  5=Excellent",
        promptTitle="Rating Scale",
    )
    dv_r.sqref = f"E5:G{4 + n_r + 1}"
    ws_r.add_data_validation(dv_r)

    for local_i, (_, row_data) in enumerate(rater_rows.iterrows()):
        r   = local_i + 5
        bg  = LIGHT if local_i % 2 == 0 else WHITE
        ws_r.row_dimensions[r].height = 60

        styled(ws_r, r, 1, row_data.name + 1, bg=bg, bold=True, align="center")  # global row #
        styled(ws_r, r, 2, row_data["tboli_source"],      bg=bg, italic=True)
        styled(ws_r, r, 3, row_data["mt_translation"],    bg=bg)
        styled(ws_r, r, 4, row_data["reference_english"], bg=bg, fg="666666", italic=True)
        for col in [5, 6, 7]:
            styled(ws_r, r, col, "", bg=YELL, align="center", bold=True, size=12)
        styled(ws_r, r, 8, "", bg=GREEN)
        styled(ws_r, r, 9, "", bg=ORANG)

    # Per-rater summary row
    SR_r = n_r + 5
    ws_r.row_dimensions[SR_r].height = 26
    ws_r.merge_cells(start_row=SR_r, start_column=1, end_row=SR_r, end_column=4)
    c = ws_r.cell(row=SR_r, column=1, value="YOUR AVERAGES")
    c.font      = Font(name="Arial", bold=True, color=WHITE, size=10)
    c.fill      = PatternFill("solid", fgColor=DARK)
    c.alignment = Alignment(horizontal="center", vertical="center")
    for col, col_letter in [(5, "E"), (6, "F"), (7, "G")]:
        fc = ws_r.cell(row=SR_r, column=col,
                       value=f"=IFERROR(AVERAGE({col_letter}5:{col_letter}{SR_r-1}),\"\")")
        fc.font          = Font(name="Arial", bold=True, color=WHITE, size=11)
        fc.fill          = PatternFill("solid", fgColor=TEAL)
        fc.alignment     = Alignment(horizontal="center", vertical="center")
        fc.number_format = "0.00"
        fc.border        = full_border
    for col in [8, 9]:
        styled(ws_r, SR_r, col, "", bg=DARK)

print(f" Per-rater sheets built (Rater 01 – Rater {NUM_RATERS:02d})")

# Sheet 2: README (for DPO pipeline)
readme = wb.create_sheet("README — DPO Pipeline")
readme.sheet_view.showGridLines = False
readme.column_dimensions["A"].width = 100

banner(readme, 1, "HOW THIS FILE FLOWS INTO DPO TRAINING", DARK, merge_to=1)
readme.row_dimensions[1].height = 30

steps = [
    ("STEP 1 — RATE",
     "Open the 'MT Samples for Rating' tab. Fill in ACCURACY, FLUENCY, CULTURAL FIT "
     "(1–5) for each row. Optionally write a preferred translation if you'd correct the MT output."),
    ("STEP 2 — SAVE",
     "Save the file as-is (keep as .xlsx) OR export the 'MT Samples for Rating' tab as CSV: "
     "File → Download → CSV. Name it: tboli_mt_for_rating_RATED.csv"),
    ("STEP 3 — UPLOAD TO COLAB",
     "In your RLHF Colab notebook, upload the rated CSV/XLSX. "
     "Set FEEDBACK_CSV = 'tboli_mt_for_rating_RATED.csv' in rlhf_dpo_training.py"),
    ("STEP 4 — DPO TRAINING",
     "Run rlhf_dpo_training.py. It reads your scores, assigns 'chosen'/'rejected' labels "
     "(avg_score ≥ 3.5 = chosen), builds preference pairs, and trains via DPO."),
    ("COLUMN REFERENCE",
     "Required by DPO script:\n"
     "  tboli_source         ← Column B (auto-filled, do not edit)\n"
     "  mt_translation       ← Column C (auto-filled, do not edit)\n"
     "  accuracy             ← Column E (YOU fill in)\n"
     "  fluency              ← Column F (YOU fill in)\n"
     "  cultural_fit         ← Column G (YOU fill in)\n"
     "  preferred_translation← Column H (optional, write your correction)\n"
     "  comments             ← Column I (optional, notes)\n"
     "  reference_english    ← Column D (for reference only, not used by DPO script)"),
    ("DPO LABEL LOGIC",
     "  avg_score = (accuracy + fluency + cultural_fit) / 3\n"
     "  If avg_score >= 3.5  →  label = 'chosen'  (reward this translation)\n"
     "  If avg_score <  3.5  →  label = 'rejected' (penalise this translation)\n"
     "  If preferred_translation is filled in → DPO uses it as 'chosen' vs MT as 'rejected'"),
]

for i, (title, desc) in enumerate(steps, 3):
    readme.row_dimensions[i].height = 90
    c1 = readme.cell(row=i, column=1,
                     value=f"{'─'*5}  {title}\n\n{desc}")
    c1.font      = Font(name="Arial", size=10,
                        bold=False, color=DARK)
    c1.fill      = PatternFill("solid", fgColor=LIGHT if i%2==0 else WHITE)
    c1.alignment = Alignment(wrap_text=True, vertical="top")

wb.save(OUTPUT_XLSX)
print(f" XLSX saved → {OUTPUT_XLSX}")



In [ ]:
# export a clean CSV (full set, for DPO script)
csv_df = sample_df[["tboli_source", "mt_translation", "reference_english"]].copy()
for col in ["accuracy", "fluency", "cultural_fit",
            "preferred_translation", "comments"]:
    csv_df[col] = ""

csv_df.to_csv(OUTPUT_CSV, index=False)
print(f" CSV saved  → {OUTPUT_CSV}")


In [ ]:
#  Download from Colab
try:
    from google.colab import files
    files.download(OUTPUT_XLSX)
    files.download(OUTPUT_CSV)
    print(" Download started — check your browser downloads folder.")
except ImportError:
    print(f"Not running in Colab — files saved locally:\n  {OUTPUT_XLSX}\n  {OUTPUT_CSV}")


In [ ]:
# Summary
print(f"""
╔══════════════════════════════════════════════════════╗
║          MT EXPORT COMPLETE                          ║
╠══════════════════════════════════════════════════════╣
║  Total rows exported : {n_total:<30}║
║  Rows per rater      : ~{rows_per_rater:<29}║
║  Number of raters    : {NUM_RATERS:<30}║
║  Pre-RLHF BLEU       : {bleu:<30.2f}║
║  Pre-RLHF chrF       : {chrf:<30.2f}║
╠══════════════════════════════════════════════════════╣
║  NEXT STEPS:                                         ║
║  1. Send each rater their tab (Rater 01, 02 …)       ║
║     OR send the whole XLSX — each rater works on     ║
║     their own numbered sheet                         ║
║  2. Raters fill columns E, F, G (scores) + H (fix)   ║
║  3. Collect back → merge → run rlhf_dpo_training.py  ║
╚══════════════════════════════════════════════════════╝
""")


# REFINING MT MODEL WITH RLHF-DPO

In [ ]:
# Install dependencies
!pip install trl transformers datasets accelerate peft sacrebleu evaluate -q


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Ensure a clean copy of the dataset from Drive
!rm -rf /content/tboli_dataset  # Remove any previous incomplete or incorrect copy
!mkdir -p /content/tboli_dataset # Ensure the target directory exists
!cp -r /content/drive/MyDrive/tboli_dataset/* /content/tboli_dataset/ # Copy contents

In [ ]:
from huggingface_hub import logout; logout()

In [ ]:
# Hugging Face Login (for pushing model if desired)
from huggingface_hub import notebook_login
notebook_login()  # Paste your HF token


In [ ]:
# Imports & config
import os, json, torch
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (MBartForConditionalGeneration,
                          MBart50TokenizerFast,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq)
from trl import DPOTrainer, DPOConfig
import evaluate


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("crslnjoyz/mt-pre-RLHF-v2")
model = AutoModelForSeq2SeqLM.from_pretrained("crslnjoyz/mt-pre-RLHF-v2")

In [ ]:
# Paths
MODEL_PATH     = "crslnjoyz/mt-pre-RLHF-v2"             # MT model
FEEDBACK_PATH  = "/content/tboli_dataset/tboli_mt_for_rating.csv"   # rated file (


OUTPUT_DIR     = "./mt-mbart-50-rlhf-dpo-tboli-lang"
HF_PUSH        = True                         # push to HuggingFace Hub


PRE_RLHF_BLEU  = 92.20
PRE_RLHF_CHRF  = 93.82

# Hyperparameters
DPO_EPOCHS     = 5       # 3–5 epochs
DPO_BETA       = 0.5     # KL-penalty coefficient (lower = more policy freedom)
LR             = 2e-5    # conservative LR for fine-tuning
BATCH_SIZE     = 2       # per-device; use gradient_accumulation for effective 8+
GRAD_ACCUM     = 1
MAX_LEN        = 128
MIN_AVG_SCORE  = 3.5     # avg_score >= threshold → "chosen"; else "rejected"

print(" Imports done | Device:", "GPU" if torch.cuda.is_available() else "CPU")


In [ ]:
# Load & aggregate feedback
def load_feedback(path: str) -> pd.DataFrame:
    """
    Handles two input formats:

    FORMAT A — Multi-sheet XLSX (from mt_inference_export.py):
      Sheets named 'Rater 01', 'Rater 02', etc.
      Each sheet has: tboli_source, mt_translation, reference_english,
                      accuracy, fluency, cultural_fit,
                      preferred_translation (optional), comments (optional)
      → Reads ALL rater sheets, stacks them, then averages scores
        per tboli_source so each sentence has one aggregated score row.

    FORMAT B — Single CSV (from Streamlit app):
      All columns in one flat file — loaded directly.
    """
    required = {"tboli_source", "mt_translation", "accuracy", "fluency", "cultural_fit"}

    # Format A: multi-sheet XLSX
    if path.endswith(".xlsx"):
        xl        = pd.ExcelFile(path)
        rater_sheets = [s for s in xl.sheet_names if s.strip().lower().startswith("rater")]

        if not rater_sheets:
            # fallback: try reading the first sheet (single-sheet XLSX)
            df = pd.read_excel(path, sheet_name=0)
            df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
        else:
            print(f"  Found {len(rater_sheets)} rater sheets: {rater_sheets}")
            frames = []
            for sheet in rater_sheets:
                df_r = pd.read_excel(path, sheet_name=sheet)
                df_r.columns = [c.strip().lower().replace(" ", "_") for c in df_r.columns]
                # Drop rows where rating columns are all empty (unrated rows)
                rating_cols = [c for c in ["accuracy","fluency","cultural_fit"]
                               if c in df_r.columns]
                df_r = df_r.dropna(subset=rating_cols, how="all")
                df_r["rater_sheet"] = sheet
                frames.append(df_r)

            if not frames:
                raise ValueError("All rater sheets are empty — no ratings found.")

            raw = pd.concat(frames, ignore_index=True)
            missing = required - set(raw.columns)
            assert not missing, f"Missing columns in XLSX: {missing}"

            # Cast score columns to numeric (raters might enter as text)
            for col in ["accuracy", "fluency", "cultural_fit"]:
                raw[col] = pd.to_numeric(raw[col], errors="coerce")

            raw = raw.dropna(subset=["accuracy", "fluency", "cultural_fit"])

            # Aggregate: average scores across raters per sentence
            # Keep preferred_translation and comments from whichever rater
            # provided them (concatenate non-empty ones)
            agg_dict = {
                "mt_translation":   "first",
                "accuracy":         "mean",
                "fluency":          "mean",
                "cultural_fit":     "mean",
            }
            if "reference_english" in raw.columns:
                agg_dict["reference_english"] = "first"
            if "preferred_translation" in raw.columns:
                # pick the longest non-empty preferred translation
                raw["preferred_translation"] = raw["preferred_translation"].fillna("").astype(str).str.strip()
                agg_dict["preferred_translation"] = lambda x: max(x, key=len)
            if "comments" in raw.columns:
                raw["comments"] = raw["comments"].fillna("").astype(str)
                agg_dict["comments"] = lambda x: " | ".join(v for v in x if v.strip())

            df = raw.groupby("tboli_source", as_index=False).agg(agg_dict)

            print(f"  Aggregated {len(raw)} individual ratings "
                  f"→ {len(df)} unique sentences")

    # Format B: CSV (Streamlit output or flat export)
    else:
        df = pd.read_csv(path)
        df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

    # Common cleaning for both formats
    missing = required - set(df.columns)
    assert not missing, f"Missing columns: {missing}"

    for col in ["accuracy", "fluency", "cultural_fit"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["tboli_source", "mt_translation",
                            "accuracy", "fluency", "cultural_fit"])
    df["tboli_source"]   = df["tboli_source"].astype(str).str.strip()
    df["mt_translation"] = df["mt_translation"].astype(str).str.strip()

    if "preferred_translation" in df.columns:
        df["preferred_translation"] = (df["preferred_translation"]
                                       .fillna("").astype(str).str.strip())

    # Compute averaged score and DPO label
    df["avg_score"] = (df["accuracy"] + df["fluency"] + df["cultural_fit"]) / 3
    df["dpo_label"] = df["avg_score"].apply(
        lambda s: "chosen" if s >= MIN_AVG_SCORE else "rejected")

    print(f"\n Feedback loaded: {len(df)} sentences")
    print(f"   Chosen (avg ≥ {MIN_AVG_SCORE}): {(df.dpo_label=='chosen').sum()}")
    print(f"   Rejected (avg < {MIN_AVG_SCORE}): {(df.dpo_label=='rejected').sum()}")
    print(f"   Avg accuracy   : {df['accuracy'].mean():.2f}")
    print(f"   Avg fluency    : {df['fluency'].mean():.2f}")
    print(f"   Avg cultural   : {df['cultural_fit'].mean():.2f}")
    return df

feedback_df = load_feedback(FEEDBACK_PATH)
feedback_df.head(3)


In [ ]:
#paper

def load_feedback(path: str) -> pd.DataFrame:
    required = {"tboli_source", "mt_translation", "accuracy", "fluency", "cultural_fit"}

    df = pd.read_csv(path)
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

    for col in ["accuracy", "fluency", "cultural_fit"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["tboli_source", "mt_translation",
                            "accuracy", "fluency", "cultural_fit"])
    df["tboli_source"]   = df["tboli_source"].astype(str).str.strip()
    df["mt_translation"] = df["mt_translation"].astype(str).str.strip()

    if "preferred_translation" in df.columns:
        df["preferred_translation"] = (df["preferred_translation"]
                                       .fillna("").astype(str).str.strip())

    df["avg_score"] = (df["accuracy"] + df["fluency"] + df["cultural_fit"]) / 3
    df["dpo_label"] = df["avg_score"].apply(
        lambda s: "chosen" if s >= MIN_AVG_SCORE else "rejected")
    return df

In [ ]:
#  Build DPO pairs (fixed)
def build_dpo_pairs(df: pd.DataFrame) -> pd.DataFrame:

    pairs = []
    has_preferred = ("preferred_translation" in df.columns and
                     (df["preferred_translation"].str.len() > 3).any())
    has_reference = ("reference_english" in df.columns and
                     df["reference_english"].notna().any())

    #  Strategy A: human preferred correction vs MT
    # Deduplicate on (tboli_source, preferred_translation) only —
    # not the full triple — so one correction per unique source is kept.
    if has_preferred:
        strat_a = (df[df["preferred_translation"].str.len() > 3]
                   .drop_duplicates(subset=["tboli_source", "preferred_translation"])
                   .copy())
        for _, row in strat_a.iterrows():
            # Only add if preferred is actually different from MT
            if row["preferred_translation"].strip() != row["mt_translation"].strip():
                pairs.append({
                    "prompt":   row["tboli_source"],
                    "chosen":   row["preferred_translation"],
                    "rejected": row["mt_translation"],
                    "strategy": "A_human_preferred",
                })
        print(f"  Strategy A (human preferred vs MT)      : {sum(1 for p in pairs if p['strategy']=='A_human_preferred'):>4} pairs")

    #  Strategy B: reference English vs MT for ALL differing rows
    # Don't limit to only rejected rows — if MT ≠ reference, the reference
    # is always the better translation regardless of rater score.
    if has_reference:
        strat_b = df[
            (df["reference_english"].notna()) &
            (df["reference_english"].astype(str).str.strip() != "") &
            (df["reference_english"].astype(str).str.strip() !=
             df["mt_translation"].astype(str).str.strip())
        ].drop_duplicates(subset=["tboli_source"]).copy()

        for _, row in strat_b.iterrows():
            pairs.append({
                "prompt":   row["tboli_source"],
                "chosen":   str(row["reference_english"]).strip(),
                "rejected": row["mt_translation"],
                "strategy": "B_reference_vs_MT",
            })
        print(f"  Strategy B (reference vs differing MT)  : {sum(1 for p in pairs if p['strategy']=='B_reference_vs_MT'):>4} pairs")

    #  Strategy C: high-score MT vs low-score MT (cross-rater)
    grouped = df.groupby("tboli_source")
    strat_c_count = 0
    for source, grp in grouped:
        high = grp[grp["dpo_label"] == "chosen"]
        low  = grp[grp["dpo_label"] == "rejected"]
        if len(high) > 0 and len(low) > 0:
            for _, h_row in high.iterrows():
                for _, l_row in low.iterrows():
                    if h_row["mt_translation"] != l_row["mt_translation"]:
                        pairs.append({
                            "prompt":   source,
                            "chosen":   h_row["mt_translation"],
                            "rejected": l_row["mt_translation"],
                            "strategy": "C_highMT_vs_lowMT",
                        })
                        strat_c_count += 1
    print(f"  Strategy C (high MT vs low MT)          : {strat_c_count:>4} pairs")

    if not pairs:
        raise ValueError(
            "No DPO pairs built. Check that score columns are filled "
            "and reference_english column exists."
        )

    pairs_df = (pd.DataFrame(pairs)
                .drop_duplicates(subset=["prompt", "chosen", "rejected"])
                .reset_index(drop=True))

    print(f"\n Total DPO pairs: {len(pairs_df)}  "
          f"(A: {(pairs_df.strategy=='A_human_preferred').sum()}  "
          f"B: {(pairs_df.strategy=='B_reference_vs_MT').sum()}  "
          f"C: {(pairs_df.strategy=='C_highMT_vs_lowMT').sum()})")
    return pairs_df

pairs_df = build_dpo_pairs(feedback_df)
pairs_df.head(5)

In [ ]:


def build_dpo_pairs(df: pd.DataFrame) -> pd.DataFrame:
    pairs         = []
    has_preferred = ("preferred_translation" in df.columns and
                     (df["preferred_translation"].str.len() > 3).any())
    has_reference = ("reference_english" in df.columns and
                     df["reference_english"].notna().any())

    # Strategy A — human correction vs. MT output
    if has_preferred:
        strat_a = (df[df["preferred_translation"].str.len() > 3]
                   .drop_duplicates(subset=["tboli_source", "preferred_translation"])
                   .copy())
        for _, row in strat_a.iterrows():
            if row["preferred_translation"].strip() != row["mt_translation"].strip():
                pairs.append({
                    "prompt":   row["tboli_source"],
                    "chosen":   row["preferred_translation"],
                    "rejected": row["mt_translation"],
                    "strategy": "A_human_preferred",
                })

    # Strategy B — reference English vs. differing MT output
    if has_reference:
        strat_b = df[
            (df["reference_english"].notna()) &
            (df["reference_english"].astype(str).str.strip() !=
             df["mt_translation"].astype(str).str.strip())
        ].drop_duplicates(subset=["tboli_source"]).copy()
        for _, row in strat_b.iterrows():
            pairs.append({
                "prompt":   row["tboli_source"],
                "chosen":   str(row["reference_english"]).strip(),
                "rejected": row["mt_translation"],
                "strategy": "B_reference_vs_MT",
            })

    # Strategy C — high-score MT vs. low-score MT
    for source, grp in df.groupby("tboli_source"):
        high = grp[grp["dpo_label"] == "chosen"]
        low  = grp[grp["dpo_label"] == "rejected"]
        for _, h in high.iterrows():
            for _, l in low.iterrows():
                if h["mt_translation"] != l["mt_translation"]:
                    pairs.append({
                        "prompt":   source,
                        "chosen":   h["mt_translation"],
                        "rejected": l["mt_translation"],
                        "strategy": "C_highMT_vs_lowMT",
                    })

    pairs_df = (pd.DataFrame(pairs)
                .drop_duplicates(subset=["prompt", "chosen", "rejected"])
                .reset_index(drop=True))
    return pairs_df

In [ ]:

# Load tokenizer and model
tokenizer = MBart50TokenizerFast.from_pretrained(
    MODEL_PATH, src_lang="tl_XX", tgt_lang="en_XX"
)
model = MBartForConditionalGeneration.from_pretrained(MODEL_PATH)
model.generation_config.forced_bos_token_id = tokenizer.lang_code_to_id["en_XX"]

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device)

print(f" Model loaded on {device}")


In [ ]:
# Prepare raw text datasets

from datasets import Dataset

n     = len(pairs_df)
split = int(n * 0.8)

# Only keep the three columns DPOTrainer needs
train_ds = Dataset.from_pandas(
    pairs_df[["prompt", "chosen", "rejected"]].iloc[:split].reset_index(drop=True)
)
val_ds = Dataset.from_pandas(
    pairs_df[["prompt", "chosen", "rejected"]].iloc[split:].reset_index(drop=True)
)

print(f" Train pairs: {len(train_ds)} | Val pairs: {len(val_ds)}")



In [ ]:
# DPO Training
import packaging.version
from trl import __version__ as TRL_VERSION

trl_new = packaging.version.parse(TRL_VERSION) >= packaging.version.parse("0.29.0")
print(f"TRL version: {TRL_VERSION}  →  using {'processing_class' if trl_new else 'tokenizer'}=")

model.tokenizer = tokenizer

steps_per_epoch = max(1, len(train_ds) // (BATCH_SIZE * GRAD_ACCUM))
total_steps     = steps_per_epoch * DPO_EPOCHS
print(f"Steps per epoch: {steps_per_epoch}  |  Total steps: {total_steps}")

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=DPO_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    beta=DPO_BETA,
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=max(1, steps_per_epoch),
    save_steps=max(1, steps_per_epoch),
    logging_steps=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    push_to_hub=HF_PUSH,
    max_length=MAX_LEN,
    **({} if trl_new else {"max_prompt_length": MAX_LEN // 2}),
)

# Build trainer kwargs depending on TRL version
trainer_kwargs = {
    "model":          model,
    "args":           dpo_config,
    "train_dataset":  train_ds,
    "eval_dataset":   val_ds,
}
if trl_new:
    trainer_kwargs["processing_class"] = tokenizer
else:
    pass

dpo_trainer = DPOTrainer(**trainer_kwargs)


In [ ]:
#paper

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=DPO_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    beta=DPO_BETA,
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=max(1, steps_per_epoch),
    save_steps=max(1, steps_per_epoch),
    logging_steps=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    push_to_hub=HF_PUSH,
    max_length=MAX_LEN,
)

trainer_kwargs = {
    "model":         model,
    "args":          dpo_config,
    "train_dataset": train_ds,
    "eval_dataset":  val_ds,
}
if trl_new:
    trainer_kwargs["processing_class"] = tokenizer
else:
    model.tokenizer = tokenizer

dpo_trainer = DPOTrainer(**trainer_kwargs)
dpo_trainer.train()

In [ ]:
dpo_trainer.train()
print("DPO training complete!")

In [ ]:
# Save & evaluate
import torch
model_to_save = dpo_trainer.model
model_to_save.lm_head.weight = torch.nn.Parameter(
    model_to_save.lm_head.weight.detach().clone()
)

# Save
dpo_trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f" Model saved to {OUTPUT_DIR}")

if HF_PUSH:
    dpo_trainer.push_to_hub()
    print("Pushed to HuggingFace Hub")

# Verify: reload and check for warnings
from transformers import MBartForConditionalGeneration
test_load = MBartForConditionalGeneration.from_pretrained(OUTPUT_DIR)
print("Reload successful — no missing key warnings above means the fix worked.")

# Final Model Evaluation

In [ ]:
!pip install jiwer evaluate sacrebleu scipy openpyxl transformers -q

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Ensure a clean copy of the dataset from Drive
!rm -rf /content/tboli_dataset  # Remove any previous incomplete or incorrect copy
!mkdir -p /content/tboli_dataset # Ensure the target directory exists
!cp -r /content/drive/MyDrive/tboli_dataset/* /content/tboli_dataset/ # Copy contents

In [ ]:
from huggingface_hub import logout; logout()

In [ ]:
# Hugging Face Login (for pushing model if desired)
from huggingface_hub import notebook_login
notebook_login()  # Paste HF token


In [ ]:
# Load Pre RLHF MT model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("crslnjoyz/mt-pre-RLHF-v2")
model = AutoModelForSeq2SeqLM.from_pretrained("crslnjoyz/mt-pre-RLHF-v2")

In [ ]:
# Load Post RLHF MT model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("crslnjoyz/mt-mbart-50-rlhf-dpo-tboli-lang")
model = AutoModelForSeq2SeqLM.from_pretrained("crslnjoyz/mt-mbart-50-rlhf-dpo-tboli-lang")

In [ ]:
# Load asr model directly
from transformers import AutoModel
model = AutoModel.from_pretrained("crslnjoyz/whisper-tboli-asr-baseline", dtype="auto")

In [ ]:
# Imports & config
import os, json, torch, warnings
import numpy as np
import pandas as pd
from scipy import stats
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.chart import BarChart, Reference
from transformers import (
    AutoModelForSpeechSeq2Seq, AutoProcessor,
    MBartForConditionalGeneration, MBart50TokenizerFast,
    pipeline,
)
import evaluate as hf_evaluate
from jiwer import wer as compute_wer, cer as compute_cer
import librosa

warnings.filterwarnings("ignore")

#  Paths
DATASET_CSV     = "/content/tboli_dataset/metadata_clean.csv"
AUDIO_DIR       = "/content"      # root folder containing audio files
AUDIO_COL       = "audio_path"                  # column in CSV with audio file paths
TRANSCRIPT_COL  = "tboli_script"                # ground-truth ASR reference column

ASR_MODEL_ID    = "crslnjoyz/asr-whisper-large-v3-tboli-lang"
PRE_RLHF_DIR = "crslnjoyz/mt-pre-RLHF-v2"           # HF hub repo name
POST_DPO_DIR = "crslnjoyz/mt-mbart-50-rlhf-dpo-tboli-lang"  # HF hub repo name

MAX_LEN         = 128
BATCH_SIZE      = 8
OUTPUT_XLSX     = "tboli_model_evaluation_report.xlsx"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Imports done | Device: {device}")



In [ ]:
# Load test set
df_full  = pd.read_csv(DATASET_CSV)
test_df  = df_full[df_full["split"] == "test"].dropna(
    subset=[TRANSCRIPT_COL, "english_script"]
).reset_index(drop=True)

print(f" Test set loaded: {len(test_df)} rows")
print(f"   Columns available: {test_df.columns.tolist()}")


In [ ]:
# Quick path sanity check
import os
sample_path = os.path.join("/content", test_df["audio_path"].iloc[0])
print(f"Resolved path : {sample_path}")
print(f"File exists   : {os.path.exists(sample_path)}")


In [ ]:
# ASR Evaluation — LoRA/PEFT correct loading
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel

#  Load base whisper-large-v3
print("Loading base Whisper large-v3 …")
base_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-large-v3",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

# Load your LoRA adapter on top of base model
print("Loading LoRA adapter from HuggingFace …")
asr_model = PeftModel.from_pretrained(base_model, ASR_MODEL_ID)
asr_model.eval()

# Load processor (language=tl, task=transcribe — same as training)
print("Loading processor …")
asr_processor = WhisperProcessor.from_pretrained(
    "openai/whisper-large-v3",
    language="tl",        # Tagalog as proxy for Tboli
    task="transcribe",
)
print(" ASR model loaded correctly (base + LoRA)")


#  Direct inference function
def transcribe(audio_path):
    audio, sr = librosa.load(audio_path, sr=16000)
    inputs    = asr_processor(
        audio, sampling_rate=16000, return_tensors="pt"
    )
    input_features = inputs["input_features"].to(
        device, dtype=torch.float16 if device == "cuda" else torch.float32
    )

    with torch.no_grad():
        predicted_ids = asr_model.generate(
            input_features,
            language="tl",
            task="transcribe",
            forced_decoder_ids=None,
        )
    return asr_processor.batch_decode(
        predicted_ids, skip_special_tokens=True
    )[0].strip().lower()


# Evaluation loop
def run_asr_evaluation(df, audio_col, transcript_col, audio_dir):
    hypotheses, references = [], []
    per_wer_l, per_cer_l   = [], []
    skipped = 0

    for idx, row in df.iterrows():
        audio_path = os.path.join(audio_dir, str(row[audio_col]))
        ref        = str(row[transcript_col]).strip().lower()

        if not os.path.exists(audio_path):
            skipped += 1
            continue
        try:
            hyp = transcribe(audio_path)
            hypotheses.append(hyp)
            references.append(ref)
            per_wer_l.append(compute_wer(ref, hyp))
            per_cer_l.append(compute_cer(ref, hyp))

            if len(hypotheses) % 50 == 0:
                print(f"  Progress: {len(hypotheses)}/{len(df)} …  "
                      f"last: [{ref}] → [{hyp}]")
        except Exception as e:
            print(f"  ✗ {os.path.basename(audio_path)}: {type(e).__name__}: {e}")
            skipped += 1

    print(f"\n  Evaluated: {len(hypotheses)} | Skipped: {skipped}")
    return hypotheses, references, per_wer_l, per_cer_l


print("\n" + "="*58)
print("  ASR EVALUATION — Whisper large-v3 + LoRA (Tboli)")
print("="*58)

asr_hyps, asr_refs, per_wer_list, per_cer_list = run_asr_evaluation(
    test_df, AUDIO_COL, TRANSCRIPT_COL, AUDIO_DIR
)

overall_wer = compute_wer(asr_refs, asr_hyps) * 100
overall_cer = compute_cer(asr_refs, asr_hyps) * 100
wer_mean    = np.mean(per_wer_list) * 100
wer_std     = np.std(per_wer_list)  * 100
cer_mean    = np.mean(per_cer_list) * 100
cer_std     = np.std(per_cer_list)  * 100

print(f"\n  WER (corpus-level) : {overall_wer:.2f}%")
print(f"  WER (mean ± std)   : {wer_mean:.2f}% ± {wer_std:.2f}%")
print(f"  CER (corpus-level) : {overall_cer:.2f}%")
print(f"  CER (mean ± std)   : {cer_mean:.2f}% ± {cer_std:.2f}%")

In [ ]:
# MT EVALUATION HELPER
bleu_metric = hf_evaluate.load("bleu")
chrf_metric = hf_evaluate.load("chrf")

def load_mt_model(model_dir):
    tok = MBart50TokenizerFast.from_pretrained(
        model_dir, src_lang="tl_XX", tgt_lang="en_XX"
    )
    mdl = MBartForConditionalGeneration.from_pretrained(model_dir)
    mdl.generation_config.forced_bos_token_id = tok.lang_code_to_id["en_XX"]
    mdl.config.use_cache = True
    mdl = mdl.to(device).eval()
    return tok, mdl

def translate_all(texts, tokenizer, model, batch_size=BATCH_SIZE):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc   = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_LEN
        ).to(device)
        with torch.no_grad():
            out = model.generate(**enc, num_beams=4, max_length=MAX_LEN)
        results += tokenizer.batch_decode(out, skip_special_tokens=True)
        print(f"  Translated {min(i+batch_size, len(texts))}/{len(texts)} …",
              end="\r")
    print()
    return results

# Calculate Pre-RLHF MT Model
def per_sentence_bleu_chrf(predictions, references):
    """Returns per-sentence BLEU and chrF scores as lists."""
    per_bleu = []
    per_chrf = []
    for pred, ref in zip(predictions, references):
        b = bleu_metric.compute(
            predictions=[pred], references=[[ref]]
        )["bleu"] * 100
        c = chrf_metric.compute(
            predictions=[pred], references=[ref]
        )["score"]
        per_bleu.append(b)
        per_chrf.append(c)
    return per_bleu, per_chrf


In [ ]:
#  PRE-RLHF MT EVALUATION ─
print("\n" + "="*58)
print("  MT EVALUATION — Pre-RLHF (Baseline)")
print("="*58)

tboli_texts = test_df["tboli_script"].tolist()
ref_texts   = test_df["english_script"].tolist()

print("Loading pre-RLHF model …")
pre_tok, pre_mdl = load_mt_model(PRE_RLHF_DIR)
print("Translating …")
pre_preds = translate_all(tboli_texts, pre_tok, pre_mdl)

pre_bleu_corpus = bleu_metric.compute(
    predictions=pre_preds, references=[[r] for r in ref_texts]
)["bleu"] * 100
pre_chrf_corpus = chrf_metric.compute(
    predictions=pre_preds, references=ref_texts
)["score"]

pre_per_bleu, pre_per_chrf = per_sentence_bleu_chrf(pre_preds, ref_texts)

print(f"\n  BLEU (corpus) : {pre_bleu_corpus:.2f}")
print(f"  BLEU (mean)   : {np.mean(pre_per_bleu):.2f} ± {np.std(pre_per_bleu):.2f}")
print(f"  chrF (corpus) : {pre_chrf_corpus:.2f}")
print(f"  chrF (mean)   : {np.mean(pre_per_chrf):.2f} ± {np.std(pre_per_chrf):.2f}")

# Free memory
del pre_mdl
torch.cuda.empty_cache() if device == "cuda" else None



In [ ]:
# POST-DPO MT EVALUATION

# Calculate Post-RLHF MT Model Evaluation
print("\n" + "="*58)
print("  MT EVALUATION — Post-DPO (RLHF-Refined)")
print("="*58)

print("Loading post-DPO model …")
post_tok, post_mdl = load_mt_model(POST_DPO_DIR)
print("Translating …")
post_preds = translate_all(tboli_texts, post_tok, post_mdl)

post_bleu_corpus = bleu_metric.compute(
    predictions=post_preds, references=[[r] for r in ref_texts]
)["bleu"] * 100
post_chrf_corpus = chrf_metric.compute(
    predictions=post_preds, references=ref_texts
)["score"]

post_per_bleu, post_per_chrf = per_sentence_bleu_chrf(post_preds, ref_texts)

print(f"\n  BLEU (corpus) : {post_bleu_corpus:.2f}")
print(f"  BLEU (mean)   : {np.mean(post_per_bleu):.2f} ± {np.std(post_per_bleu):.2f}")
print(f"  chrF (corpus) : {post_chrf_corpus:.2f}")
print(f"  chrF (mean)   : {np.mean(post_per_chrf):.2f} ± {np.std(post_per_chrf):.2f}")

del post_mdl
torch.cuda.empty_cache() if device == "cuda" else None


In [ ]:
#  PAIRED T-TESTS
# Compares pre-RLHF vs post-DPO (RLHF Refined) on a per-sentence basis.
# scipy paired t-test, reporting mean, SD, p-value, 95% CI.

print("\n" + "="*58)
print("  STATISTICAL ANALYSIS — Paired T-Tests")
print("="*58)

def paired_ttest(pre_scores, post_scores, metric_name):
    """
    Two-tailed paired t-test between pre and post scores.
    Returns dict with all values needed for paper reporting.
    """
    pre   = np.array(pre_scores)
    post  = np.array(post_scores)
    diffs = post - pre                        # positive = improvement

    d_mean = np.mean(diffs)
    d_std  = np.std(diffs, ddof=1)
    n      = len(diffs)
    se     = d_std / np.sqrt(n)

    t_stat, p_val = stats.ttest_rel(pre, post)

    # 95% confidence interval on the mean difference
    ci = stats.t.interval(0.95, df=n-1, loc=d_mean, scale=se)

    print(f"\n  [{metric_name}]")
    print(f"   Pre-RLHF  mean ± SD : {np.mean(pre):.4f} ± {np.std(pre, ddof=1):.4f}")
    print(f"   Post-DPO  mean ± SD : {np.mean(post):.4f} ± {np.std(post, ddof=1):.4f}")
    print(f"   Mean diff (post-pre): {d_mean:+.4f}")
    print(f"   t-statistic         : {t_stat:.4f}")
    print(f"   p-value             : {p_val:.6f}  {'★ significant (p<0.05)' if p_val < 0.05 else '(not significant)'}")
    print(f"   95% CI              : [{ci[0]:.4f}, {ci[1]:.4f}]")
    print(f"   n (sentence pairs)  : {n}")

    return {
        "metric":       metric_name,
        "pre_mean":     round(float(np.mean(pre)), 4),
        "pre_std":      round(float(np.std(pre, ddof=1)), 4),
        "post_mean":    round(float(np.mean(post)), 4),
        "post_std":     round(float(np.std(post, ddof=1)), 4),
        "mean_diff":    round(float(d_mean), 4),
        "t_statistic":  round(float(t_stat), 4),
        "p_value":      round(float(p_val), 6),
        "ci_lower":     round(float(ci[0]), 4),
        "ci_upper":     round(float(ci[1]), 4),
        "n":            n,
        "significant":  bool(p_val < 0.05),
    }

ttest_bleu = paired_ttest(pre_per_bleu,  post_per_bleu,  "BLEU")
ttest_chrf = paired_ttest(pre_per_chrf,  post_per_chrf,  "chrF")

# WER: no pre/post since ASR was not changed by DPO
# WER/CER as standalone ASR metrics (no t-test needed)
print(f"\n  [WER / CER] — ASR model unchanged by DPO, no paired comparison needed.")
print(f"   WER: {overall_wer:.2f}% | CER: {overall_cer:.2f}%")



In [ ]:
#  BUILD XLSX REPORT
print("\n\nBuilding XLSX evaluation report …")

#  Style helpers
DARK    = "1A2E4A"
TEAL    = "2E86AB"
GREEN   = "27AE60"
RED     = "E74C3C"
GOLD    = "F39C12"
LIGHT   = "E8F4F8"
WHITE   = "FFFFFF"
LGREY   = "F5F5F5"

thin    = Side(style="thin",   color="CCCCCC")
med     = Side(style="medium", color=TEAL)
bdr     = Border(top=thin, bottom=thin, left=thin, right=thin)
bdr_med = Border(top=Side(style="medium", color=DARK),
                 bottom=Side(style="medium", color=DARK),
                 left=Side(style="medium", color=DARK),
                 right=Side(style="medium", color=DARK))

def cell(ws, r, c, val="", bg=WHITE, fg=DARK, bold=False, size=10,
         align="left", wrap=False, num_fmt=None, italic=False, border=bdr):
    x = ws.cell(row=r, column=c, value=val)
    x.font      = Font(name="Arial", bold=bold, color=fg, size=size, italic=italic)
    x.fill      = PatternFill("solid", fgColor=bg)
    x.alignment = Alignment(horizontal=align, vertical="center", wrap_text=wrap)
    x.border    = border
    if num_fmt: x.number_format = num_fmt
    return x

def hdr_row(ws, row, cols_vals, bg=DARK, fg=WHITE, size=10, height=30):
    ws.row_dimensions[row].height = height
    for c, val in enumerate(cols_vals, 1):
        cell(ws, row, c, val, bg=bg, fg=fg, bold=True, size=size,
             align="center", wrap=True)

def banner(ws, row, text, bg=DARK, fg=WHITE, size=14, merge_to=8, height=38):
    ws.merge_cells(start_row=row, start_column=1,
                   end_row=row, end_column=merge_to)
    ws.row_dimensions[row].height = height
    c = ws.cell(row=row, column=1, value=text)
    c.font      = Font(name="Arial", bold=True, color=fg, size=size)
    c.fill      = PatternFill("solid", fgColor=bg)
    c.alignment = Alignment(horizontal="center", vertical="center")

wb = Workbook()

# SHEET 1 · SUMMARY DASHBOARD
ws1 = wb.active
ws1.title = "Summary Dashboard"
ws1.sheet_view.showGridLines = False

col_widths = [22, 18, 18, 18, 18, 18, 18, 18]
for i, w in enumerate(col_widths, 1):
    ws1.column_dimensions[get_column_letter(i)].width = w

ws1.row_dimensions[1].height = 8
banner(ws1, 2, "TBOLI E2E ASR+MT · MODEL EVALUATION REPORT", DARK, size=15, merge_to=8)
banner(ws1, 3,
       f"Test Set: {len(test_df)} rows  |  ASR: {ASR_MODEL_ID}  |  "
       f"MT Pre-RLHF: {PRE_RLHF_DIR}  |  MT Post-DPO: {POST_DPO_DIR}",
       TEAL, size=9, merge_to=8, height=22)

# ASR section
ws1.row_dimensions[5].height = 20
banner(ws1, 5, "ASR PERFORMANCE  (Whisper — unchanged by DPO)", TEAL, size=11, merge_to=8, height=24)

hdr_row(ws1, 6, ["Metric", "Value", "Mean ± SD", "Notes", "", "", "", ""], height=28)

asr_rows = [
    ("WER (%)",
     f"{overall_wer:.2f}",
     f"{wer_mean:.2f} ± {wer_std:.2f}",
     "Lower is better. Corpus-level WER reported."),
    ("CER (%)",
     f"{overall_cer:.2f}",
     f"{cer_mean:.2f} ± {cer_std:.2f}",
     "Lower is better. Character-level error rate."),
]
for i, (metric, val, ms, note) in enumerate(asr_rows, 7):
    bg = LIGHT if i % 2 == 1 else WHITE
    ws1.row_dimensions[i].height = 22
    cell(ws1, i, 1, metric, bg=bg, bold=True)
    cell(ws1, i, 2, val,    bg=bg, align="center")
    cell(ws1, i, 3, ms,     bg=bg, align="center")
    ws1.merge_cells(start_row=i, start_column=4, end_row=i, end_column=8)
    cell(ws1, i, 4, note,   bg=bg, italic=True, fg="666666")

# MT section
ws1.row_dimensions[10].height = 8
banner(ws1, 11, "MT PERFORMANCE  (MBart — Pre-RLHF vs Post-DPO v2)", TEAL, size=11, merge_to=8, height=24)

hdr_row(ws1, 12,
        ["Metric", "Pre-RLHF", "Post-DPO v2", "Change", "% Change",
         "Significant?", "Direction", "Notes"],
        height=28)

bleu_delta  = post_bleu_corpus - pre_bleu_corpus
chrf_delta  = post_chrf_corpus - pre_chrf_corpus
bleu_pct    = (bleu_delta / pre_bleu_corpus * 100) if pre_bleu_corpus > 0 else 0
chrf_pct    = (chrf_delta / pre_chrf_corpus * 100) if pre_chrf_corpus > 0 else 0

mt_rows = [
    ("BLEU", pre_bleu_corpus, post_bleu_corpus, bleu_delta, bleu_pct,
     ttest_bleu["significant"], "Higher is better",
     "Corpus-level BLEU on test set"),
    ("chrF", pre_chrf_corpus, post_chrf_corpus, chrf_delta, chrf_pct,
     ttest_chrf["significant"], "Higher is better",
     "Primary metric for morphologically rich languages"),
]
for i, (metric, pre, post, delta, pct, sig, direction, note) in enumerate(mt_rows, 13):
    bg = LIGHT if i % 2 == 1 else WHITE
    ws1.row_dimensions[i].height = 24
    delta_fg = GREEN if delta >= 0 else RED
    cell(ws1, i, 1, metric,              bg=bg, bold=True)
    cell(ws1, i, 2, round(pre, 2),       bg=bg, align="center", num_fmt="0.00")
    cell(ws1, i, 3, round(post, 2),      bg=bg, align="center", num_fmt="0.00")
    cell(ws1, i, 4, round(delta, 2),     bg=bg, align="center", num_fmt="+0.00;-0.00",
         fg=delta_fg, bold=True)
    cell(ws1, i, 5, round(pct, 2),       bg=bg, align="center", num_fmt="+0.00%;-0.00%")
    cell(ws1, i, 6, "Yes ★" if sig else "No", bg=bg, align="center",
         fg=GREEN if sig else RED, bold=sig)
    cell(ws1, i, 7, direction,           bg=bg, italic=True, fg="666666")
    cell(ws1, i, 8, note,                bg=bg, italic=True, fg="666666")

# Interpretation note
ws1.row_dimensions[16].height = 8
ws1.merge_cells(start_row=17, start_column=1, end_row=17, end_column=8)
ws1.row_dimensions[17].height = 70
c = ws1.cell(row=17, column=1,
             value=(
                 "INTERPRETATION: The DPO-refined model (Post-DPO v2) achieved a BLEU of "
                 f"{post_bleu_corpus:.2f} and chrF of {post_chrf_corpus:.2f} on the held-out test set, "
                 f"representing changes of {bleu_delta:+.2f} BLEU and {chrf_delta:+.2f} chrF over the "
                 "pre-RLHF baseline. The pre-RLHF model's high baseline accuracy (93.5% exact match "
                 "with references) created a ceiling effect limiting available DPO preference pairs to 57. "
                 "chrF is the primary metric for Tboli given its morphological richness. "
                 "ASR performance (WER/CER) was unaffected by DPO as the Whisper model was not retrained."
             ))
c.font      = Font(name="Arial", size=9, italic=True, color="444444")
c.fill      = PatternFill("solid", fgColor="FFF9C4")
c.alignment = Alignment(wrap_text=True, vertical="top")
c.border    = bdr



# SHEET 2: PAIRED T-TEST RESULTS

ws2 = wb.create_sheet("Paired T-Test Results")
ws2.sheet_view.showGridLines = False

for i, w in enumerate([22, 16, 16, 16, 16, 16, 16, 16, 16, 16, 18], 1):
    ws2.column_dimensions[get_column_letter(i)].width = w

ws2.row_dimensions[1].height = 8
banner(ws2, 2, "PAIRED T-TEST RESULTS — Pre-RLHF vs Post-DPO v2", merge_to=11, height=36)
banner(ws2, 3,
       "Two-tailed paired t-test (scipy.stats.ttest_rel) | α = 0.05 | "
       f"n = {ttest_bleu['n']} sentence pairs",
       TEAL, size=9, merge_to=11, height=22)

hdr_row(ws2, 5,
        ["Metric", "Pre Mean", "Pre SD", "Post Mean", "Post SD",
         "Mean Diff", "t-stat", "p-value", "95% CI Lower", "95% CI Upper",
         "Significant?"],
        height=32)
ws2.row_dimensions[5].height = 32

# Formula for t-test from paper: t = d̄ / (s_d / √n)
ws2.row_dimensions[4].height = 20
ws2.merge_cells(start_row=4, start_column=1, end_row=4, end_column=11)
c = ws2.cell(row=4, column=1,
             value="Formula: t = d̄ / (Sd / √n)  where d̄ = mean of differences, "
                   "Sd = standard deviation of differences, n = number of pairs")
c.font      = Font(name="Arial", size=9, italic=True, color=DARK)
c.fill      = PatternFill("solid", fgColor=LIGHT)
c.alignment = Alignment(horizontal="center", vertical="center")

for i, ttest in enumerate([ttest_bleu, ttest_chrf], 6):
    bg = LIGHT if i % 2 == 0 else WHITE
    ws2.row_dimensions[i].height = 26
    sig = ttest["significant"]
    diff_fg = GREEN if ttest["mean_diff"] >= 0 else RED

    cell(ws2, i, 1,  ttest["metric"],     bg=bg, bold=True)
    cell(ws2, i, 2,  ttest["pre_mean"],   bg=bg, align="center", num_fmt="0.0000")
    cell(ws2, i, 3,  ttest["pre_std"],    bg=bg, align="center", num_fmt="0.0000")
    cell(ws2, i, 4,  ttest["post_mean"],  bg=bg, align="center", num_fmt="0.0000")
    cell(ws2, i, 5,  ttest["post_std"],   bg=bg, align="center", num_fmt="0.0000")
    cell(ws2, i, 6,  ttest["mean_diff"],  bg=bg, align="center",
         num_fmt="+0.0000;-0.0000", fg=diff_fg, bold=True)
    cell(ws2, i, 7,  ttest["t_statistic"],bg=bg, align="center", num_fmt="0.0000")
    cell(ws2, i, 8,  ttest["p_value"],    bg=bg, align="center", num_fmt="0.000000",
         fg=GREEN if sig else RED, bold=sig)
    cell(ws2, i, 9,  ttest["ci_lower"],   bg=bg, align="center", num_fmt="0.0000")
    cell(ws2, i, 10, ttest["ci_upper"],   bg=bg, align="center", num_fmt="0.0000")
    cell(ws2, i, 11, "Yes ★" if sig else "No", bg=bg, align="center",
         fg=GREEN if sig else RED, bold=sig)

# Interpretation block
ws2.merge_cells(start_row=9, start_column=1, end_row=9, end_column=11)
ws2.row_dimensions[9].height = 60
c = ws2.cell(row=9, column=1,
             value=(
                 "HOW TO REPORT IN PAPER: "
                 f"A paired t-test was conducted to compare pre-RLHF and post-DPO MT performance "
                 f"on {ttest_bleu['n']} test sentence pairs. "
                 f"For BLEU: t({ttest_bleu['n']-1}) = {ttest_bleu['t_statistic']:.4f}, "
                 f"p = {ttest_bleu['p_value']:.6f}, 95% CI [{ttest_bleu['ci_lower']:.4f}, {ttest_bleu['ci_upper']:.4f}]. "
                 f"For chrF: t({ttest_chrf['n']-1}) = {ttest_chrf['t_statistic']:.4f}, "
                 f"p = {ttest_chrf['p_value']:.6f}, 95% CI [{ttest_chrf['ci_lower']:.4f}, {ttest_chrf['ci_upper']:.4f}]. "
                 f"{'Both metrics showed statistically significant differences.' if ttest_bleu['significant'] and ttest_chrf['significant'] else 'Results were not statistically significant, consistent with the small DPO preference dataset (57 pairs).'}"
             ))
c.font      = Font(name="Arial", size=9, italic=True, color="1A2E4A")
c.fill      = PatternFill("solid", fgColor="FFF9C4")
c.alignment = Alignment(wrap_text=True, vertical="top")
c.border    = bdr



# SHEET 3: SENTENCE-LEVEL DETAIL
ws3 = wb.create_sheet("Sentence-Level Detail")
ws3.sheet_view.showGridLines = False
ws3.freeze_panes = "A4"

for i, w in enumerate([6, 34, 34, 34, 14, 14, 14, 14, 14], 1):
    ws3.column_dimensions[get_column_letter(i)].width = w

banner(ws3, 1, "SENTENCE-LEVEL MT EVALUATION — Test Set", merge_to=9, height=32)
banner(ws3, 2,
       "Per-sentence BLEU and chrF for pre-RLHF and post-DPO v2 models",
       TEAL, size=9, merge_to=9, height=20)

hdr_row(ws3, 3,
        ["#", "Tboli Source", "Reference English",
         "Post-DPO Translation",
         "Pre BLEU", "Post BLEU", "BLEU Δ",
         "Pre chrF", "Post chrF"],
        height=32)

for i, (src, ref, post, pb, pob, pc, poc) in enumerate(
        zip(tboli_texts, ref_texts, post_preds,
            pre_per_bleu, post_per_bleu,
            pre_per_chrf, post_per_chrf), 4):
    bg  = LIGHT if i % 2 == 0 else WHITE
    ws3.row_dimensions[i].height = 40
    delta_b = pob - pb
    delta_fg = GREEN if delta_b >= 0 else RED

    cell(ws3, i, 1, i-3,   bg=bg, align="center", bold=True)
    cell(ws3, i, 2, src,   bg=bg, italic=True, wrap=True)
    cell(ws3, i, 3, ref,   bg=bg, fg="444444", wrap=True)
    cell(ws3, i, 4, post,  bg=bg, wrap=True)
    cell(ws3, i, 5, round(pb,  2), bg=bg, align="center", num_fmt="0.00")
    cell(ws3, i, 6, round(pob, 2), bg=bg, align="center", num_fmt="0.00")
    cell(ws3, i, 7, round(delta_b, 2), bg=bg, align="center",
         num_fmt="+0.00;-0.00", fg=delta_fg, bold=True)
    cell(ws3, i, 8, round(pc,  2), bg=bg, align="center", num_fmt="0.00")
    cell(ws3, i, 9, round(poc, 2), bg=bg, align="center", num_fmt="0.00")

# Totals row
sr = len(tboli_texts) + 4
ws3.row_dimensions[sr].height = 24
for c_idx, val in [
    (1, "AVG"), (5, f"=AVERAGE(E4:E{sr-1})"), (6, f"=AVERAGE(F4:F{sr-1})"),
    (7, f"=AVERAGE(G4:G{sr-1})"), (8, f"=AVERAGE(H4:H{sr-1})"),
    (9, f"=AVERAGE(I4:I{sr-1})")
]:
    c = ws3.cell(row=sr, column=c_idx, value=val)
    c.font      = Font(name="Arial", bold=True, color=WHITE, size=10)
    c.fill      = PatternFill("solid", fgColor=TEAL)
    c.alignment = Alignment(horizontal="center", vertical="center")
    c.number_format = "0.00" if c_idx > 1 else "General"
    c.border    = bdr

ws3.merge_cells(start_row=sr, start_column=1, end_row=sr, end_column=4)
ws3.cell(row=sr, column=1).value = "AVERAGES (auto-calculated)"


# SHEET 4: ASR DETAIL
ws4 = wb.create_sheet("ASR Detail")
ws4.sheet_view.showGridLines = False
ws4.freeze_panes = "A4"

for i, w in enumerate([6, 34, 34, 14, 14, 14], 1):
    ws4.column_dimensions[get_column_letter(i)].width = w

banner(ws4, 1, "ASR SENTENCE-LEVEL EVALUATION — Whisper Tboli Baseline", merge_to=6, height=32)
banner(ws4, 2, "WER and CER per utterance on test set", TEAL, size=9, merge_to=6, height=20)

hdr_row(ws4, 3,
        ["#", "Reference Transcript", "ASR Hypothesis", "WER (%)", "CER (%)", "Match?"],
        height=30)

for i, (ref, hyp, w_score, c_score) in enumerate(
        zip(asr_refs, asr_hyps, per_wer_list, per_cer_list), 4):
    bg  = LIGHT if i % 2 == 0 else WHITE
    ws4.row_dimensions[i].height = 40
    match = w_score == 0.0

    cell(ws4, i, 1, i-3,          bg=bg, align="center", bold=True)
    cell(ws4, i, 2, ref,          bg=bg, italic=True, wrap=True)
    cell(ws4, i, 3, hyp,          bg=bg, wrap=True)
    cell(ws4, i, 4, round(w_score*100, 2), bg=bg, align="center",
         num_fmt="0.00", fg=GREEN if match else (RED if w_score > 0.5 else DARK))
    cell(ws4, i, 5, round(c_score*100, 2), bg=bg, align="center", num_fmt="0.00")
    cell(ws4, i, 6, "✓" if match else "✗", bg=bg, align="center",
         fg=GREEN if match else RED, bold=True, size=12)

sr4 = len(asr_refs) + 4
ws4.row_dimensions[sr4].height = 24
for c_idx, val in [
    (1, "AVG"), (4, f"=AVERAGE(D4:D{sr4-1})"), (5, f"=AVERAGE(E4:E{sr4-1})")
]:
    c = ws4.cell(row=sr4, column=c_idx, value=val)
    c.font      = Font(name="Arial", bold=True, color=WHITE)
    c.fill      = PatternFill("solid", fgColor=TEAL)
    c.alignment = Alignment(horizontal="center", vertical="center")
    c.number_format = "0.00" if c_idx > 1 else "General"
    c.border    = bdr
ws4.merge_cells(start_row=sr4, start_column=1, end_row=sr4, end_column=3)
ws4.cell(row=sr4, column=1).value = "AVERAGES"


# Save
wb.save(OUTPUT_XLSX)
print(f"\n XLSX report saved → {OUTPUT_XLSX}")


In [ ]:
# Download from Colab
try:
    from google.colab import files
    files.download(OUTPUT_XLSX)
    print(" Download started.")
except ImportError:
    print(f"Not in Colab — file saved locally: {OUTPUT_XLSX}")


In [ ]:
# Final console summary
print(f"""
╔══════════════════════════════════════════════════════════════╗
║           COMPLETE EVALUATION SUMMARY                        ║
╠══════════════════════════════════════════════════════════════╣
║  TEST SET SIZE    : {len(test_df):<42}║
╠══════════════════════════════════════════════════════════════╣
║  ASR (Whisper — unchanged by DPO)                           ║
║  WER              : {overall_wer:<6.2f}%  (mean: {wer_mean:.2f}% ± {wer_std:.2f}%)          ║
║  CER              : {overall_cer:<6.2f}%  (mean: {cer_mean:.2f}% ± {cer_std:.2f}%)          ║
╠══════════════════════════════════════════════════════════════╣
║  MT TRANSLATION                                              ║
║  {'Metric':<10} {'Pre-RLHF':>10} {'Post-DPO':>10} {'Change':>10}              ║
║  {'BLEU':<10} {pre_bleu_corpus:>10.2f} {post_bleu_corpus:>10.2f} {bleu_delta:>+10.2f}              ║
║  {'chrF':<10} {pre_chrf_corpus:>10.2f} {post_chrf_corpus:>10.2f} {chrf_delta:>+10.2f}              ║
╠══════════════════════════════════════════════════════════════╣
║  PAIRED T-TESTS                                              ║
║  BLEU: t={ttest_bleu['t_statistic']:>8.4f}  p={ttest_bleu['p_value']:>10.6f}  {'★ significant' if ttest_bleu['significant'] else 'not significant':<18}║
║  chrF: t={ttest_chrf['t_statistic']:>8.4f}  p={ttest_chrf['p_value']:>10.6f}  {'★ significant' if ttest_chrf['significant'] else 'not significant':<18}║
╠══════════════════════════════════════════════════════════════╣
║  XLSX REPORT      : {OUTPUT_XLSX:<42}║
╚══════════════════════════════════════════════════════════════╝
""")
